# 03 - Claim a bounded work batch

This dispatcher notebook grants one bounded, runtime-compatible item batch without exceeding the configured active-worker limit. Claims are conditional Delta updates. A retry with the same `DISPATCHER_ID` returns its remaining unstarted claims instead of allocating more work while any owned lease is active.

**After importing into Fabric:** On the configuration code cell, select **... -> Toggle parameter cell** and confirm the parameter indicator. Then attach and pin `people_counter_<environment>` as this notebook's default Lakehouse.

In [ ]:
DISPATCHER_ID = ""
MAX_CONCURRENT_WORKERS = 4
CLAIM_LIMIT = 4
LEASE_MINUTES = 30
DATABASE = ""
TABLE_PREFIX = "people_counter"

In [ ]:
from datetime import datetime, timedelta, timezone
import json
import random
import re
import time
import uuid

from people_counter.fabric_control import ControlWriter
from people_counter.fabric_events import process_worker_events
import notebookutils
from pyspark.sql import SparkSession, functions as F


IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")
dispatcher_id = DISPATCHER_ID.strip()
if not dispatcher_id:
    raise ValueError("DISPATCHER_ID must be a stable pipeline run ID")
max_workers = int(MAX_CONCURRENT_WORKERS)
claim_limit = int(CLAIM_LIMIT)
lease_minutes = int(LEASE_MINUTES)
if max_workers < 1 or claim_limit < 1 or lease_minutes < 5:
    raise ValueError("Worker and claim limits must be positive; LEASE_MINUTES must be at least 5")
database = DATABASE.strip()
prefix = TABLE_PREFIX.strip()
if database and IDENTIFIER.fullmatch(database) is None:
    raise ValueError("DATABASE is not a valid identifier")
if IDENTIFIER.fullmatch(prefix) is None:
    raise ValueError("TABLE_PREFIX is not a valid identifier")


def table(suffix: str) -> str:
    value = f"{prefix}_{suffix}"
    return f"{database}.{value}" if database else value


spark_candidate = globals().get("spark")
if not isinstance(spark_candidate, SparkSession):
    raise RuntimeError("A Fabric Spark session is required")
spark_session = spark_candidate
spark_session.conf.set("spark.sql.session.timeZone", "UTC")
writer = ControlWriter(spark_session, table("control_writer"))
DeltaTable = writer.tables
process_worker_events(spark_session, writer, table_prefix=prefix, database=database)
work_table = table("video_work")
attempts_table = table("video_attempts")
dispatcher_leases_table = table("dispatcher_leases")
now = datetime.now(timezone.utc)
lease_expires = now + timedelta(minutes=lease_minutes)
active_states = ["LEASED", "STAGING", "RUNNING", "WRITING"]


def retry_delta(operation, attempts: int = 6) -> None:
    for number in range(attempts):
        try:
            operation()
            return
        except Exception as error:
            message = f"{type(error).__name__}: {error}".lower()
            retryable = any(token in message for token in ("concurrent", "conflict", "changedexception"))
            if not retryable or number == attempts - 1:
                raise
            time.sleep((2 ** number) * 0.25 + random.random() * 0.5)


lock_expires = now + timedelta(minutes=35)
lock_owner = f"{dispatcher_id}:{uuid.uuid4().hex}"
lock_source = spark_session.createDataFrame(
    [("global", lock_owner, now, lock_expires)],
    "lock_name string, owner_id string, acquired_at timestamp, expires_at timestamp",
)
retry_delta(
    lambda: (
        DeltaTable.forName(spark_session, dispatcher_leases_table)
        .alias("t")
        .merge(lock_source.alias("s"), "t.lock_name = s.lock_name")
        .whenMatchedUpdateAll(condition="t.expires_at <= current_timestamp()")
        .whenNotMatchedInsertAll()
        .execute()
    )
)
def assert_dispatcher_lock() -> None:
    lock_rows = (
        spark_session.table(dispatcher_leases_table)
        .where(
            (F.col("lock_name") == "global")
            & (F.col("owner_id") == lock_owner)
            & (F.col("expires_at") > F.current_timestamp())
        )
        .limit(2)
        .collect()
    )
    if len(lock_rows) != 1:
        raise RuntimeError("Dispatcher claim mutex ownership was lost or expired")


def renew_dispatcher_lock() -> None:
    renewed_until = datetime.now(timezone.utc) + timedelta(minutes=35)
    retry_delta(
        lambda: DeltaTable.forName(spark_session, dispatcher_leases_table).update(
            condition=(
                (F.col("lock_name") == "global")
                & (F.col("owner_id") == lock_owner)
                & (F.col("expires_at") > F.current_timestamp())
            ),
            set={"expires_at": F.lit(renewed_until)},
        )
    )
    assert_dispatcher_lock()


def release_dispatcher_lock() -> None:
    retry_delta(
        lambda: DeltaTable.forName(spark_session, dispatcher_leases_table).delete(
            (F.col("lock_name") == "global") & (F.col("owner_id") == lock_owner)
        )
    )


def run_with_lock_cleanup(operation):
    try:
        return operation()
    except Exception:
        release_dispatcher_lock()
        raise


run_with_lock_cleanup(assert_dispatcher_lock)

owned_active = (
    spark_session.table(work_table)
    .where(
        (F.col("lease_dispatcher_id") == dispatcher_id)
        & F.col("status").isin(active_states)
        & (F.col("lease_expires_at") > F.current_timestamp())
    )
    .select(
        "work_id",
        F.col("lease_owner_attempt_id").alias("attempt_id"),
        "status",
    )
)
owned_rows = run_with_lock_cleanup(owned_active.limit(1).collect)

if owned_rows:
    claimed = owned_active.where(F.col("status") == "LEASED").select("work_id", "attempt_id")
    active_before = None
else:
    active_before = run_with_lock_cleanup(
        lambda: spark_session.table(work_table)
        .where(F.col("status").isin(active_states) & (F.col("lease_expires_at") > F.current_timestamp()))
        .where(F.col("lease_dispatcher_id").isNotNull())
        .select("lease_dispatcher_id")
        .distinct()
        .count()
    )
    available = max(0, max_workers - active_before)
    take = claim_limit if available > 0 else 0
    if take == 0:
        claimed = spark_session.createDataFrame([], "work_id string, attempt_id string")
    else:
        eligible = (
            spark_session.table(work_table)
            .where(
                F.col("status").isin("QUEUED", "RETRY_WAIT")
                & (F.col("attempt_count") < F.col("max_attempts"))
                & (F.col("not_before_at").isNull() | (F.col("not_before_at") <= F.lit(now)))
                & (F.col("lease_expires_at").isNull() | (F.col("lease_expires_at") <= F.lit(now)))
            )
            .orderBy(F.col("priority").desc(), F.col("queue_entered_at").asc(), F.col("work_id"))
        )
        first_candidate = run_with_lock_cleanup(
            lambda: eligible.select(
                F.coalesce("runtime_sha256", "config_sha256").alias("runtime_sha256")
            ).limit(1).collect()
        )
        if first_candidate:
            selected_runtime_sha256 = first_candidate[0].runtime_sha256
            candidate_rows = run_with_lock_cleanup(
                lambda: eligible
                .where(
                    F.coalesce("runtime_sha256", "config_sha256")
                    == F.lit(selected_runtime_sha256)
                )
                .limit(take)
                .select("work_id", "capture_date")
                .collect()
            )
        else:
            selected_runtime_sha256 = None
            candidate_rows = []
        source = spark_session.createDataFrame(
            [
                (row.work_id, row.capture_date, uuid.uuid4().hex, dispatcher_id, lease_expires)
                for row in candidate_rows
            ],
            "work_id string, capture_date date, attempt_id string, dispatcher_id string, lease_expires timestamp",
        )
        run_with_lock_cleanup(renew_dispatcher_lock)
        try:
            retry_delta(
                lambda: (
                    DeltaTable.forName(spark_session, work_table)
                    .alias("t")
                    .merge(source.alias("s"), "t.work_id = s.work_id AND t.capture_date = s.capture_date")
                    .whenMatchedUpdate(
                        condition=(
                            "t.status IN ('QUEUED', 'RETRY_WAIT') AND "
                            "t.attempt_count < t.max_attempts AND "
                            "(t.not_before_at IS NULL OR t.not_before_at <= current_timestamp()) AND "
                            "(t.lease_expires_at IS NULL OR t.lease_expires_at <= current_timestamp())"
                        ),
                        set={
                            "status": "'LEASED'",
                            "attempt_count": "t.attempt_count + 1",
                            "lease_owner_attempt_id": "s.attempt_id",
                            "lease_dispatcher_id": "s.dispatcher_id",
                            "lease_acquired_at": "current_timestamp()",
                            "lease_expires_at": "s.lease_expires",
                            "last_heartbeat_at": "current_timestamp()",
                            "not_before_at": "NULL",
                        },
                    )
                    .execute()
                )
            )
        except Exception:
            release_dispatcher_lock()
            raise
        claimed = (
            source.alias("s")
            .join(
                spark_session.table(work_table).alias("w"),
                (F.col("s.work_id") == F.col("w.work_id"))
                & (F.col("s.capture_date") == F.col("w.capture_date"))
                & (F.col("s.attempt_id") == F.col("w.lease_owner_attempt_id")),
                "inner",
            )
            .where(
                (F.col("w.lease_dispatcher_id") == dispatcher_id)
                & (F.col("w.status") == "LEASED")
                & (F.col("w.lease_expires_at") > F.current_timestamp())
            )
            .select(F.col("s.work_id"), F.col("s.attempt_id"))
        )

claimed_count = run_with_lock_cleanup(claimed.count)
claimed_runtime_count = run_with_lock_cleanup(
    lambda: claimed.join(spark_session.table(work_table), "work_id")
    .select(F.coalesce("runtime_sha256", "config_sha256").alias("runtime_sha256"))
    .distinct().count()
)
if claimed_count and claimed_runtime_count != 1:
    claims_to_release = claimed.withColumn("dispatcher_id", F.lit(dispatcher_id))
    run_with_lock_cleanup(lambda: retry_delta(
        lambda: (
            DeltaTable.forName(spark_session, work_table)
            .alias("t")
            .merge(claims_to_release.alias("s"), "t.work_id = s.work_id")
            .whenMatchedUpdate(
                condition=(
                    "t.status = 'LEASED' AND "
                    "t.lease_dispatcher_id = s.dispatcher_id AND "
                    "t.lease_owner_attempt_id = s.attempt_id"
                ),
                set={
                    "status": "'QUEUED'",
                    "attempt_count": "greatest(t.attempt_count - 1, 0)",
                    "lease_owner_attempt_id": "NULL",
                    "lease_dispatcher_id": "NULL",
                    "lease_acquired_at": "NULL",
                    "lease_expires_at": "NULL",
                    "queue_entered_at": "current_timestamp()",
                },
            )
            .execute()
        )
    ))
    release_dispatcher_lock()
    raise RuntimeError("Claimed worker batch must contain exactly one compatible model runtime")

attempt_rows = (
    claimed.alias("c")
    .join(spark_session.table(work_table).alias("w"), "work_id")
    .select(
        F.col("c.attempt_id"),
        F.col("work_id"),
        F.lit(dispatcher_id).alias("dispatcher_id"),
        F.lit(None).cast("string").alias("pipeline_run_id"),
        F.lit(None).cast("string").alias("activity_run_id"),
        F.lit(None).cast("string").alias("fabric_job_instance_id"),
        F.lit(None).cast("string").alias("worker_execution_id"),
        F.lit(None).cast("string").alias("sdk_version"),
        F.lit(None).cast("string").alias("bundle_manifest_sha256"),
        F.col("w.config_sha256"),
        F.lit("LEASED").alias("status"),
        F.col("w.lease_acquired_at").alias("claimed_at"),
        F.lit(None).cast("timestamp").alias("staging_started_at"),
        F.lit(None).cast("timestamp").alias("inference_started_at"),
        F.lit(None).cast("timestamp").alias("writing_started_at"),
        F.lit(None).cast("timestamp").alias("completed_at"),
        F.col("w.last_heartbeat_at"),
        F.lit(None).cast("string").alias("input_sha256"),
        F.lit(None).cast("long").alias("source_size_bytes"),
        F.lit(None).cast("double").alias("source_duration_seconds"),
        F.lit(None).cast("double").alias("source_fps"),
        F.lit(None).cast("long").alias("total_source_frames"),
        F.lit(None).cast("long").alias("processed_frames"),
        F.lit(None).cast("double").alias("effective_sample_fps"),
        F.lit(None).cast("double").alias("processing_seconds"),
        F.lit(None).cast("long").alias("distinct_people"),
        F.lit(None).cast("long").alias("line_in_count"),
        F.lit(None).cast("long").alias("line_out_count"),
        F.lit(None).cast("boolean").alias("retryable"),
        F.lit(None).cast("string").alias("error_category"),
        F.lit(None).cast("string").alias("error_type"),
        F.lit(None).cast("string").alias("error_message"),
        F.col("w.capture_date"),
    )
)
try:
    retry_delta(
        lambda: (
            DeltaTable.forName(spark_session, attempts_table)
            .alias("t")
            .merge(attempt_rows.alias("s"), "t.attempt_id = s.attempt_id AND t.capture_date = s.capture_date")
            .whenNotMatchedInsertAll()
            .execute()
        )
    )
except Exception:
    release_dispatcher_lock()
    raise
claimed_attempts = (
    claimed.alias("c")
    .join(
        spark_session.table(attempts_table).alias("a"),
        (F.col("c.work_id") == F.col("a.work_id"))
        & (F.col("c.attempt_id") == F.col("a.attempt_id")),
        "inner",
    )
    .join(
        spark_session.table(work_table).alias("w"),
        F.col("c.work_id") == F.col("w.work_id"),
        "inner",
    )
    .where(
        (F.col("a.dispatcher_id") == dispatcher_id)
        & (F.col("a.status") == "LEASED")
        & (F.col("a.capture_date") == F.col("w.capture_date"))
        & F.col("a.config_sha256").eqNullSafe(F.col("w.config_sha256"))
    )
    .select(F.col("c.work_id"), F.col("c.attempt_id"))
)
verified_attempt_count = run_with_lock_cleanup(claimed_attempts.count)
if verified_attempt_count != claimed_count:
    release_dispatcher_lock()
    raise RuntimeError(
        f"Claim batch has {claimed_count} work leases but {verified_attempt_count} matching attempts"
    )
items = [
    row.asDict(recursive=True)
    for row in run_with_lock_cleanup(lambda: claimed.orderBy("work_id").collect())
]
outcome = {
    "dispatcher_id": dispatcher_id,
    "active_before": active_before,
    "claimed_count": claimed_count,
    "items": items,
}
release_dispatcher_lock()
print(json.dumps(outcome, sort_keys=True))

In [ ]:
notebookutils.notebook.exit(json.dumps(outcome, sort_keys=True))